In [67]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [68]:
#Load env variables
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [69]:
#Create an API client
from anthropic import Anthropic
import os

client = Anthropic()
model = os.environ["ANTHROPIC_MODEL"]
print(model)

anthropic.claude-4-5-haiku


In [70]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [71]:
import json

def generate_dataset():
  prompt = """
    Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
      {
        "task": "Description of task",
        "format": python, json or regex
      },
      ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
    """

  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "```json")
  text = chat(messages, stop_sequences=["```"])
  return json.loads(text)

In [72]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [73]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

*Respond only with Puthon, Json, or a plain Regex
*Do not add any comments or commentry or explanation
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [74]:
import re

def extract_json(text):
    """Extrai JSON do texto, removendo markdown se necessário."""
    text = text.strip()
    # tenta extrair de bloco ```json ... ```
    match = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if match:
        text = match.group(1).strip()
    return text

def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    
    Respond with raw JSON only, no markdown, no code fences, no explanation.
    Example response shape:
    {{
        "strengths": ["..."],
        "weaknesses": ["..."],
        "reasoning": "...",
        "score": 8
    }}
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    
    system = "Respond only with raw JSON. No markdown, no code fences, no explanation."
    eval_text = chat(messages, system=system)
    
    cleaned = extract_json(eval_text)
    
    if not cleaned:
        raise ValueError(f"Resposta vazia ou sem JSON. Raw: {repr(eval_text[:300])}")
    
    return json.loads(cleaned)

In [75]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    if format == "regex":
        return validate_regex(response)
    if format == "python":
        return validate_python(response)

In [76]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case,output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    syntax_score = grade_syntax(output,test_case)
    
    score = (syntax_score + model_score)/2
    
    return {
        "output": output,
        "test_case": test_case,
        "model_score": model_score,
        "syntax_score": syntax_score,
        "score": score,
        "reasoning": reasoning
    }

In [77]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [78]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [79]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef parse_s3_uri(uri):\n    pattern = r's3://([^/]+)(?:/(.+)/([^/]+))?$|s3://([^/]+)/([^/]+)$'\n    match = re.match(pattern, uri)\n    \n    if match:\n        groups = match.groups()\n        if groups[0]:  # First pattern matched\n            bucket = groups[0]\n            if groups[1]:\n                folder_path = groups[1]\n                file_name = groups[2]\n            else:\n                folder_path = \"\"\n                file_name = \"\"\n        else:  # Second pattern matched\n            bucket = groups[3]\n            folder_path = \"\"\n            file_name = groups[4]\n    else:\n        bucket = folder_path = file_name = None\n    \n    return {\n        \"bucket\": bucket,\n        \"folder_path\": folder_path,\n        \"file_name\": file_name\n    }\n\nuri = \"s3://my-bucket/path/to/file.txt\"\nprint(parse_s3_uri(uri))\n",
    "test_case": {
      "task": "Parse an AWS S3 object key to extract the bucket name, folder pat